# Day 1 — Meet the Machines 🤖

> **Mission briefing:** You passed the entrance exam, so here's your reward: a walk through the Lab's equipment room. Four finished AI machines are humming on the benches — one that **sees**, one that **reads moods**, one that **writes stories**, and a **sorting hat** that files text into any categories you invent. Today you just get to *play* with them. There's a catch, though: by Demo Day you will have **rebuilt every single one yourself**.

**Previously in the Lab:** you were the Data Detective — you can read data. Now meet the machines that devour it.

**Today you will:**
- See the **map of AI** — how machine learning, deep learning, and the "superpowers" fit together
- Run four **pretrained** models: image recognition, sentiment, text generation, zero-shot sorting
- Try to **fool** them, bend them, and break them (that's how you learn what they can't do)
- Meet the **AI Studio** — the app you'll assemble one tab per day

*These models were trained on giant datasets using enormous computers. You won't train them today — you'll borrow them, the way real researchers stand on each other's shoulders.*

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════╗
# ║ SETUP — run me first! (identical in every Lab notebook)           ║
# ╚═══════════════════════════════════════════════════════════════════╝
import os, sys, pathlib

IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("SMOKE_TEST", "0") == "1"   # tiny fast run for automated tests

REPO_URL = "https://github.com/A-Halimi/ai-studio-internship.git"  # instructor: set once

if IN_COLAB:
    if not pathlib.Path("/content/ai-studio-internship").exists():
        !git clone -q {REPO_URL} /content/ai-studio-internship
    %cd /content/ai-studio-internship/notebooks/day01
    %pip -q install transformers==5.13.0

HERE = pathlib.Path.cwd()
REPO = HERE.parents[1]                       # .../notebooks/dayNN -> repo root
DATA_DIR = pathlib.Path(os.environ.get("COURSE_DATA_DIR", REPO / "data"))
SAMPLES_DIR = REPO / "data_samples"          # small datasets shipped with the repo
MODELS_DIR = REPO / "ai_studio" / "models"   # where your Studio modules unlock
for p in (REPO / "data", MODELS_DIR):
    p.mkdir(parents=True, exist_ok=True)

import random
import numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED)
print(f"✅ Setup done | Colab: {IN_COLAB} | Smoke test: {SMOKE}")
print(f"   data: {DATA_DIR}")

In [ ]:
import torch
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    print(f"⚡ Running on GPU: {torch.cuda.get_device_name(0)}")
else:
    print("🐢 No GPU found — running on CPU. Everything still works, just slower.")

## 🗺️ The map of AI

People throw around "AI", "machine learning", and "deep learning" as if they mean the same thing. They don't — they **nest**, like Russian dolls:

- **Artificial Intelligence** — any program that does something we'd call "smart."
- **Machine Learning** — a *kind* of AI that isn't hand-coded with rules; it **learns patterns from data**. (Yesterday you saw the patterns; ML is teaching a machine to find them.)
- **Deep Learning** — machine learning with **neural networks** stacked many layers deep. It powers almost every jaw-dropping AI you've heard of.

And there are **three ways** a machine can learn:

- **Supervised** — learn from *labeled* examples ("this penguin is a Gentoo"). ← Day 2
- **Unsupervised** — find structure with *no* labels ("these things clump together"). ← Day 3
- **Reinforcement** — learn by *trial, reward, and error*, like training a dog. ← Day 9

Run the next cell for the map.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 5))

# Left: AI contains ML contains Deep Learning (nested boxes)
axL.set_xlim(0, 10); axL.set_ylim(0, 10); axL.axis("off")
for x, y, w, h, label, color, fs in [
    (0.5, 0.5, 9.0, 9.0, "Artificial Intelligence", "#e8f0fe", 13),
    (1.6, 1.6, 6.8, 6.3, "Machine Learning", "#c6dafc", 12),
    (2.8, 2.6, 4.4, 3.9, "Deep Learning", "#8ab4f8", 12),
]:
    axL.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02",
                                 fc=color, ec="#222", lw=1.5))
    axL.text(x + w / 2, y + h - 0.5, label, ha="center", va="top",
             fontsize=fs, fontweight="bold")
axL.text(5.0, 3.9, "neural networks\nwith many layers", ha="center", va="center",
         fontsize=10, style="italic")
axL.set_title("AI  ⊃  Machine Learning  ⊃  Deep Learning", fontweight="bold")

# Right: the three ways to learn + two superpowers
axR.set_xlim(0, 10); axR.set_ylim(0, 10); axR.axis("off")
axR.set_title("How machines learn", fontweight="bold")
for i, (name, desc, color) in enumerate([
    ("Supervised", "learn from labeled examples", "#a5d6a7"),
    ("Unsupervised", "find structure, no labels", "#fff59d"),
    ("Reinforcement", "learn by trial and reward", "#ef9a9a"),
]):
    y = 8.2 - i * 2.3
    axR.add_patch(FancyBboxPatch((0.5, y - 0.9), 9.0, 1.6, boxstyle="round,pad=0.02",
                                 fc=color, ec="#222"))
    axR.text(5.0, y + 0.15, name, ha="center", fontsize=12, fontweight="bold")
    axR.text(5.0, y - 0.5, desc, ha="center", fontsize=9)
axR.text(5.0, 0.6, "Two superpowers you'll build:   Vision (images)  +  Language (text)",
         ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

## 👁️ Superpower 1 — Vision

First up: a machine that **looks at a photo and names what's in it**. Ours is **ResNet-18**, a neural network trained on **ImageNet** — 1.2 million photos across 1000 categories (dogs, castles, teapots, you name it).

We don't have to train it — we just **download the finished brain** and use it. Two details matter:

- `.eval()` puts the network in "just answer, don't learn" mode.
- `weights.transforms()` is the exact **preprocessing** (resize, crop, normalize) the model expects. Feed it anything else and it gets confused — models are picky eaters.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.datasets import load_sample_image
from PIL import Image

weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights).eval().to(DEVICE)
preprocess = weights.transforms()          # the model's required input pipeline
categories = weights.meta["categories"]    # the 1000 label names
print(f"ResNet-18 ready — it knows {len(categories)} categories.")


def classify_top5(img_array):
    """Return the model's 5 most confident labels for one image (a NumPy array)."""
    img = Image.fromarray(img_array)
    batch = preprocess(img).unsqueeze(0).to(DEVICE)   # 1 image -> a batch of 1
    with torch.no_grad():
        logits = model(batch)
    probs = torch.softmax(logits, dim=1)[0]           # turn scores into probabilities
    top_p, top_i = probs.topk(5)
    return [(categories[i], p.item()) for i, p in zip(top_i, top_p)]


def show_top5(img_array, title):
    preds = classify_top5(img_array)
    labels = [name for name, _ in preds][::-1]
    scores = [p for _, p in preds][::-1]
    fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(10, 4))
    ax_img.imshow(img_array); ax_img.axis("off"); ax_img.set_title(title)
    ax_bar.barh(labels, scores)
    ax_bar.set_xlim(0, 1)
    ax_bar.set_xlabel("Confidence")
    ax_bar.set_title("Top-5 guesses")
    plt.tight_layout(); plt.show()
    return preds

In [ ]:
china = load_sample_image("china.jpg")     # a photo that ships with scikit-learn
_ = show_top5(china, "china.jpg")

In [ ]:
flower = load_sample_image("flower.jpg")
_ = show_top5(flower, "flower.jpg")

### 🔬 Your turn (mini-task)

Look at the two bar charts. **In a sentence or two, jot down what the model got right and what it got wrong.** Was it confidently correct? Confidently *wrong*? Notice that it never says "I don't know" — it always spreads its bet across the labels it was trained on, even for a photo that matches none of them well. Keeping models honest about their own uncertainty is a real research problem.

## 💬 Superpower 2 — Reading the mood

Next machine: **sentiment analysis**. It reads a sentence and decides whether the feeling behind it is **positive** or **negative**, with a confidence score.

The `transformers` library gives us a one-liner called a **pipeline**: name the task, name the model, and it downloads and wires everything up. Ours is a **DistilBERT** model fine-tuned on movie reviews.

In [ ]:
from transformers import pipeline

mood = pipeline("text-classification",
                model="distilbert-base-uncased-finetuned-sst-2-english",
                device=0 if DEVICE.type == "cuda" else -1)

sentences = [
    "I absolutely loved this movie, what a ride!",
    "This is the worst day I have ever had.",
    "The food was okay, nothing special.",
]
for s in sentences:
    r = mood(s)[0]
    print(f"{r['label']:8} ({r['score']:.2f})   {s}")

### 🧪 Exercise 1 — Fool the mood reader

These models learn from patterns of words, so they struggle with things humans read instantly — like **sarcasm**.

- Write a sentence that *sounds* positive word-for-word but really means the opposite, and store it in `sarcastic`.
- **Goal:** get the model to answer **POSITIVE** for something you clearly meant as a complaint.

In [ ]:
# ==================== YOUR CODE HERE ====================
### HINT: pile on happy words for an unhappy situation
### HINT: e.g. "Oh great, my flight got cancelled again. I just LOVE airports."
...
result = mood(sarcastic)[0]
print(f"{result['label']} ({result['score']:.2f})   {sarcastic}")

## ✍️ Superpower 3 — The story writer

This machine **generates text**. Give **GPT-2** the start of a sentence and it keeps going, one word at a time, each word guessed from everything before it. (Yes — it's a distant, much smaller ancestor of the models you talk to today.)

A couple of dials:

- `max_new_tokens` — how many words to add.
- `temperature` — the "creativity" knob. Low = safe and repetitive, high = wild. We use `0.9` for some spice.

In [ ]:
storyteller = pipeline("text-generation", model="gpt2",
                       device=0 if DEVICE.type == "cuda" else -1)

MAX_NEW = 20 if SMOKE else 40
prompt = "In the year 3000, the last robot on Earth discovered"
torch.manual_seed(SEED)          # reproducible demo
out = storyteller(prompt, max_new_tokens=MAX_NEW, do_sample=True,
                  temperature=0.9, pad_token_id=50256)
print(out[0]["generated_text"])

### 🧪 Exercise 2 — Be the author

Swap in **your own** prompt and see what the machine dreams up.

- Set `my_prompt` to any opening you like — a fairy tale, a news headline, a recipe.
- Run it a few times: because `do_sample=True`, you get a different story every time.

In [ ]:
# ==================== YOUR CODE HERE ====================
### HINT: anything goes — the funnier the opener, the funnier the result
...
torch.manual_seed(SEED)          # reproducible demo
out = storyteller(my_prompt, max_new_tokens=MAX_NEW, do_sample=True,
                  temperature=0.9, pad_token_id=50256)
print(out[0]["generated_text"])

## 🎩 Superpower 4 — The sorting hat

The last machine is the cleverest trick. **Zero-shot classification** sorts text into categories **it was never trained on**. You invent the labels on the spot, and it figures out which one fits best.

"Zero-shot" means zero training examples for your specific categories — it reasons about the *meaning* of the words instead.

In [ ]:
sorter = pipeline("zero-shot-classification",
                  model="typeform/distilbert-base-uncased-mnli",
                  device=0 if DEVICE.type == "cuda" else -1)

text = "I spent all weekend building a robot that sorts my socks by color."
labels = ["technology", "sports", "cooking", "art"]
res = sorter(text, candidate_labels=labels)
for lbl, score in zip(res["labels"], res["scores"]):
    print(f"{lbl:12} {score:.2f}")

### 🧪 Exercise 3 — Invent the categories

Give the sorting hat your own text and your own labels.

- Set `my_text` to any sentence, and `my_labels` to 3–5 categories it might fall into.
- **Try to trip it up:** a sentence that could belong to two categories at once is the interesting case.

In [ ]:
my_text = "The championship match went to overtime and the crowd went wild."
# ==================== YOUR CODE HERE ====================
### HINT: pick categories the text might plausibly belong to
...
res = sorter(my_text, candidate_labels=my_labels)
print(f"Text: {my_text}\n")
for lbl, score in zip(res["labels"], res["scores"]):
    print(f"{lbl:12} {score:.2f}")

## 🧪 This is the mission: your AI Studio

Everything you just played with was built by *other* people. Over the next two weeks you'll build your **own** versions — and they'll live inside one app: the **AI Studio**.

The Studio has one tab per capability. Each starts **🔒 locked**, and you unlock it by finishing that day's notebook (which trains and saves the model the tab needs):

| Day | Tab | What you'll build |
|----|-----|-------------------|
| 2 | 🐧 Prediction Machine | classify penguins from measurements |
| 3 | 🎨 Pattern Finder | discover groups in data with *no* labels |
| 5 | ✍️ Digit Reader | recognize handwritten digits |
| 6 | 🕵️ Photo Detective | recognize objects in photos |
| 7 | 🪨📄✂️ RPS Arena | play rock–paper–scissors through the camera |
| 8 | 💬 Language Lab | understand and generate text |
| 9 | 🎮 Game Master | an agent that learns to win a game |

Want a peek? From the repo root run:

```
python ai_studio/app.py
```

Today **every tab shows 🔒** — you haven't built anything yet. That padlock wall *is* the mission. By **Demo Day (Day 10)** you'll open the app and every lock will be gone. 🔓

> **📦 A note on the first run.** Outside the course's Docker container, the very first time you load these models your computer downloads roughly **1.5 GB** of weights from the internet, so give it a minute. Inside the Lab's container everything is **pre-baked** — no download, no waiting. Either way, you only pay this cost once.

## 🏁 Checkpoint

You've met the machines. In a single afternoon you:

- ✅ Placed AI, machine learning, and deep learning on the same map
- ✅ Ran a **vision** model that names objects in photos
- ✅ Ran a **sentiment** model — and fooled it with sarcasm
- ✅ Made a **language** model write, and a **zero-shot** model sort
- ✅ Saw the **AI Studio** you're about to fill, one locked tab at a time

The big secret: none of these are magic. Each is a pattern-finder trained on data — the same idea you'll master step by step.

> **Tomorrow (Day 2):** you stop borrowing and start **building**. You'll train your own classifier on yesterday's penguins and unlock the first tab of your Studio — **🐧 Prediction Machine**. 🔓